# Brazil Property Prices: Data Understanding

### Objective

Examine the structure, variables, data types, missing values, and data quality of the Brazilian real estate datasets.

In [15]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd

pd.set_option("display.max_columns", None)

# Define the path to our first data file
path_1 = "../data/raw/brasil-real-estate-1.csv"
print(f"We will load data from: '{path_1}'")

We will load data from: '../data/raw/brasil-real-estate-1.csv'


In [21]:
# Preview the dataset
(pd.read_csv(path_1).head())

,property_type,place_with_parent_names,region,lat-lon,area_m2,price_usd
0,apartment,|Brasil|Alagoas|Maceió|,Northeast,"-9.6443051,-35.7088142",110.0,"$187,230.85"
1,apartment,|Brasil|Alagoas|Maceió|,Northeast,"-9.6430934,-35.70484",65.0,"$81,133.37"
2,house,|Brasil|Alagoas|Maceió|,Northeast,"-9.6227033,-35.7297953",211.0,"$154,465.45"
3,apartment,|Brasil|Alagoas|Maceió|,Northeast,"-9.622837,-35.719556",99.0,"$146,013.20"
4,apartment,|Brasil|Alagoas|Maceió|,Northeast,"-9.654955,-35.700227",55.0,"$101,416.71"


In [22]:
# Preview the end of the dataset
(pd.read_csv(path_1).tail())

,property_type,place_with_parent_names,region,lat-lon,area_m2,price_usd
12829,apartment,|Brasil|Pernambuco|Recife|,Northeast,"-8.056418,-34.909309",91.0,"$174,748.79"
12830,apartment,|Brasil|Pernambuco|Recife|,Northeast,"-8.1373477,-34.909181",115.0,"$115,459.02"
12831,apartment,|Brasil|Pernambuco|Recife|Boa Viagem|,Northeast,"-8.1136717,-34.896252",76.0,"$137,302.62"
12832,apartment,|Brasil|Pernambuco|Recife|Boa Viagem|,Northeast,NaN,130.0,"$234,038.56"
12833,apartment,|Brasil|Pernambuco|Recife|Boa Viagem|,Northeast,"-8.0578381,-34.882897",99.0,"$168,507.77"


## Initial Data Assessment & Data Anomalies

Upon inspecting the head and tail of the dataset, several structural anomalies and data cleaning issues are immediately apparent:

* **String formatting in `price_usd`:** The price column contains currency symbols ($) and commas (,), meaning pandas is treating it as an object (string) instead of a numeric float. This will prevent any mathematical analysis until cleaned.
* **Combined geographical strings in `place_with_parent_names`:** Location details are piped together (e.g., `|Brasil|Alagoas|Maceió|`). This column needs to be split into separate country, state, and city columns.
* **Merged coordinates in `lat-lon`:** Latitude and longitude are combined into a single string separated by a comma, and row 12832 already shows a missing value here.

In [23]:
# Check dataset schema and non-null counts
(pd.read_csv(path_1).info())

<class 'pandas.DataFrame'>
RangeIndex: 12834 entries, 0 to 12833
Data columns (total 6 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   property_type            12834 non-null  str    
 1   place_with_parent_names  12834 non-null  str    
 2   region                   12834 non-null  str    
 3   lat-lon                  11551 non-null  str    
 4   area_m2                  12834 non-null  float64
 5   price_usd                12834 non-null  str    
dtypes: float64(1), str(5)
memory usage: 601.7 KB


### Key Insights from `.info()`

* **Data Types:** The `price_usd` and `lat-lon` columns are stored as text strings rather than numeric values, which will prevent mathematical analysis and mapping until converted.
* **Missing Data:** The `lat-lon` column is missing coordinates for **1,283** rows (11,551 non-null out of 12,834).


In [24]:
# Quantify total missing values per column
(pd.read_csv(path_1).isnull().sum())

property_type                 0
place_with_parent_names       0
region                        0
lat-lon                    1283
area_m2                       0
price_usd                     0
dtype: int64

In [25]:
# Check missing coordinates by Property Type
(
    pd.read_csv(path_1)
    .assign(is_missing=lambda df: df["lat-lon"].isnull())
    .groupby("property_type")["is_missing"]
    .agg(["sum", "mean"])
)

,sum,mean
property_type,,
apartment,1068,0.099878
house,215,0.100420


In [26]:
# Check missing coordinates by Region
(
    pd.read_csv(path_1)
    .assign(is_missing=lambda df: df["lat-lon"].isnull())
    .groupby("region")["is_missing"]
    .agg(["sum", "mean"])
)

,sum,mean
region,,
Central-West,143,0.093464
North,36,0.106195
Northeast,457,0.100683
South,273,0.096912
Southeast,374,0.103630


## Missing Data Distribution Analysis Summary

To determine if dropping the missing coordinate rows would bias our analysis, the missing `lat-lon` data was aggregated across categories:

* **Uniform Distribution:** Missing values are evenly distributed at approximately **10%** across all property types (`apartment`: 9.9%, `house`: 10.0%) and all Brazilian regions (~9.3% to 10.6%).
* **Statistical Context:** The data is confirmed to be **Missing Completely at Random (MCAR)**. 

### Strategic Pivot: Why We Are Choosing Spatial Imputation Over Deletion
While an MCAR status means dropping these rows would not technically introduce systemic geographic bias, it would force us to completely wipe out **1,283 valid records** (exactly 10% of our sample size). This data loss reduces our model's sample size, limits its statistical power, and deletes 1,283 perfectly structured `price_usd` and `area_m2` data points.

To satisfy the strict non-null requirements of downstream Machine Learning models (like Linear Regression) without suffering severe data volume loss, we will utilize **Spatial Imputation**. Since we retain descriptive text location fallbacks in `place_with_parent_names`, we can mathematically project missing points onto the **median coordinate of their respective cities/neighborhoods**. This preserves 100% of our matrix rows while repairing the feature gap.

---

## Data Cleaning & Transformation Plan (Machine Learning Deployment)

Based on our refined strategy, we will implement an explicit data cleaning pipeline optimized for machine learning compatibility. 

### 1. Numeric Conversions
* **`price_usd`**: Strip the currency symbol (`$`) and remove commas (`,`). Cast the resulting string into a numeric float data type to serve as our continuous target variable ($y$).

### 2. Feature Extraction & Engineering
* **`lat-lon`**: Split the raw comma-separated coordinate strings into independent numerical float columns: `latitude` and `longitude`.
* **`place_with_parent_names`**: Parse the pipe-delimited values (`|`) to extract clean, distinct structural categorical columns for `state` and `city` (ignoring the static `Brasil` element). These text fields will be preserved for encoding transformations later.

### 3. Missing Data Strategy: Spatial Imputation
* Instead of dropping records, we will fill missing `latitude` and `longitude` coordinate points using the computed **median coordinate grouping of the specific city** extracted from the dataset. 

### 4. Dimensionality & Cleanup
* Drop the original `lat-lon` and `place_with_parent_names` raw text fields to remove structural data duplication and free up memory allocation prior to matrix modeling.


In [79]:
# Complete cleaning pipeline with fallback spatial imputation
df1 = (
    pd.read_csv(path_1)
    .assign(
        # Convert "$187,230.85" -> 187230.85 (Float)
        price_usd=lambda x: x["price_usd"]
            .str.replace("$", "", regex=False)
            .str.replace(",", "", regex=False)
            .astype(float),

        # Split text geo string into separate row float arrays
        lat=lambda x: x["lat-lon"]
            .str.split(",", expand=True)[0]
            .astype(float),
        lon=lambda x: x["lat-lon"]
            .str.split(",", expand=True)[1]
            .astype(float),

        # Extract cleam categorical columns
        # (index 2 is state, index 3 is city)
        state=lambda x: x["place_with_parent_names"]
            .str.split("|", expand=True)[2],
        city=lambda x: x["place_with_parent_names"]
            .str.split("|", expand=True)[3]
    )

    # Spatial Imputation using group-levele transform medians
    .assign(
        lat=lambda x: x["lat"]
            .fillna(
                x
                    .groupby("city")["lat"]
                    .transform("median")
            ),
        lon=lambda x: x["lon"]
            .fillna(
                x
                    .groupby("city")["lon"]
                    .transform("median")
            )
    )

    # Fallback Imputation: Fill any remaining edge
    # using state medians
    .assign(
        lat=lambda x: x["lat"]
            .fillna(
                x
                    .groupby("state")["lat"]
                    .transform("median")
            ),
        lon=lambda x: x["lon"]
            .fillna(
                x
                    .groupby("state")["lon"]
                    .transform("median")
            )
    )

    # Final structural cleanup
    .drop(columns=["lat-lon", "place_with_parent_names"])
)

# Verify the fully preserved machine learning matrix
print(f"Preseverd matrix shape: {df1.shape}")
print(f"'price_usd' dtype: {df1["price_usd"].dtype}")
print(f"'Lat' missing values: {df1["lat"].isnull().sum()}")
print(f"'Lon' missing values: {df1["lon"].isnull().sum()}")

Preseverd matrix shape: (12834, 8)
'price_usd' dtype: float64
'Lat' missing values: 0
'Lon' missing values: 0


In [80]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 12834 entries, 0 to 12833
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   property_type  12834 non-null  str    
 1   region         12834 non-null  str    
 2   area_m2        12834 non-null  float64
 3   price_usd      12834 non-null  float64
 4   lat            12834 non-null  float64
 5   lon            12834 non-null  float64
 6   state          12834 non-null  str    
 7   city           12834 non-null  str    
dtypes: float64(4), str(4)
memory usage: 802.3 KB
